# Семинар 2. Своя среда в Gymnasium: GridWorld, обёртки, Cross-Entropy

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IlyaChichkanov/Reinforcement-learning/blob/main/02-environments/seminar/seminar.ipynb)

План:

1. Каркас среды: пространства, `reset`, `step` — заполняем по шагам
2. Проверка `check_env`, текстовый и графический `render`
3. Случайный и ручной агенты; маска действий
4. Обёртки: лимит шагов, своя награда, статистика эпизодов; потенциальный shaping своими руками
5. Cross-Entropy из недели 1 на своей среде; что происходит на скользком полу
6. Ловушка наивного shaping: агент собирает подсказки вместо цели
7. Частичная наблюдаемость на CartPole: прячем скорость и возвращаем её стеком кадров
8. Что дальше: домашнее задание

Ячейки с `# TODO` заполняете сами; ниже каждой — проверка, которая должна пройти.

In [ ]:
# Если ноутбук открыт в Google Colab: ставим недостающие пакеты. Локально (после uv sync) ячейка ничего не делает.
import importlib.util, subprocess, sys
if importlib.util.find_spec("gymnasium") is None or importlib.util.find_spec("pygame") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium[classic-control]"], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env

## 1. Каркас среды

Договоримся о карте: строки, `S` — старт, `G` — цель, `#` — стена, `.` — пол. Состояние — номер клетки `row * n_cols + col`, действия `0..3` = ←, ↓, →, ↑. С вероятностью `slip` действие заменяется на случайное. Награда: 1 при приходе в цель, иначе 0; эпизод заканчивается в цели.

Заполните `reset` и `step`. Подсказки:

* `super().reset(seed=seed)` создаёт `self.np_random` — используйте **его**, а не `np.random`, иначе среда не будет воспроизводимой;
* шаг в стену или за границу оставляет агента на месте;
* `step` возвращает ровно пять значений: `obs, reward, terminated, truncated, info`.

In [ ]:
class GridWorldEnv(gym.Env):
    metadata = {"render_modes": ["ansi", "rgb_array"], "render_fps": 4}
    MOVES = {0: (0, -1), 1: (1, 0), 2: (0, 1), 3: (-1, 0)}   # ←, ↓, →, ↑  как (dr, dc)
    ARROWS = "←↓→↑"
    DEFAULT_LAYOUT = ["S....#", ".##..#", "...#..", ".#..#.", ".#.#..", "....#G"]

    def __init__(self, layout=None, slip=0.0, render_mode=None):
        self.layout = [list(row) for row in (layout or self.DEFAULT_LAYOUT)]
        self.n_rows, self.n_cols = len(self.layout), len(self.layout[0])
        self.slip = slip
        self.render_mode = render_mode
        self.observation_space = spaces.Discrete(self.n_rows * self.n_cols)
        self.action_space = spaces.Discrete(4)
        self.start = self._find("S")
        self.goal = self._find("G")
        self.pos = self.start

    def _find(self, char):
        for r, row in enumerate(self.layout):
            for c, cell in enumerate(row):
                if cell == char:
                    return (r, c)
        raise ValueError(f"на карте нет клетки {char!r}")

    def _obs(self):
        return self.pos[0] * self.n_cols + self.pos[1]

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # TODO: вернуть агента на старт и вернуть (obs, info)
        raise NotImplementedError

    def step(self, action):
        # TODO: 1) с вероятностью self.slip заменить action случайным (через self.np_random)
        #       2) сдвинуть агента, если клетка свободна и внутри поля
        #       3) terminated = агент в цели; reward = 1.0 если terminated иначе 0.0
        #       4) вернуть (obs, reward, terminated, False, {})
        raise NotImplementedError

    def render(self):
        if self.render_mode == "ansi":
            return "\n".join("".join("A" if (r, c) == self.pos else ch for c, ch in enumerate(row))
                             for r, row in enumerate(self.layout))
        if self.render_mode == "rgb_array":
            return self._render_rgb()

    def _render_rgb(self):
        colors = {".": "#f4f4f4", "#": "#555555", "S": "#cfe3f7", "G": "#a9dfa9"}
        fig, ax = plt.subplots(figsize=(2.6, 2.6 * self.n_rows / self.n_cols))
        for r, row in enumerate(self.layout):
            for c, ch in enumerate(row):
                ax.add_patch(plt.Rectangle((c, r), 1, 1, color=colors[ch], ec="white"))
        ax.add_patch(plt.Circle((self.pos[1] + 0.5, self.pos[0] + 0.5), 0.3, color="#4C72B0"))
        ax.set_xlim(0, self.n_cols); ax.set_ylim(self.n_rows, 0); ax.set_aspect("equal"); ax.axis("off")
        fig.tight_layout(pad=0); fig.canvas.draw()
        img = np.asarray(fig.canvas.buffer_rgba())[:, :, :3].copy()
        plt.close(fig)
        return img

In [ ]:
# Проверка 1: базовое поведение.
env = GridWorldEnv()
obs, info = env.reset(seed=0)
assert obs == 0 and isinstance(info, dict)
obs, r, term, trunc, info = env.step(2)          # → из старта
assert obs == 1 and r == 0.0 and not term and not trunc
obs, *_ = env.step(3)                            # ↑ в стену поля: остаёмся
assert obs == 1
env.reset(seed=0)
obs, *_ = env.step(1); obs, *_ = env.step(1)     # ↓ ↓
assert obs == 12, obs
print("OK: reset/step ведут себя правильно")

## 2. Проверка интерфейса и отрисовка

`check_env` из Gymnasium — обязательный первый тест любой среды. Он ловит неправильные типы, невоспроизводимый `seed`, лишние значения из `step`.

In [ ]:
check_env(GridWorldEnv())
print("check_env: ok")

env = GridWorldEnv(render_mode="ansi")
env.reset(seed=0)
print(env.render())

env = GridWorldEnv(render_mode="rgb_array")
env.reset(seed=0)
plt.imshow(env.render()); plt.axis("off"); plt.show()

## 3. Случайный и ручной агенты

Случайный агент — базовая линия: любая политика, которую мы обучим, должна быть лучше него. Ручной агент — таблица «клетка → действие», написанная руками: убедитесь, что вы сами умеете решать задачу, прежде чем заставлять агента.

Заполните `manual_policy` так, чтобы агент доходил до цели на карте по умолчанию. Достаточно задать действия для клеток, через которые проходит маршрут.

In [ ]:
def run_episode(env, policy, seed=0, max_steps=100):
    """policy(obs) -> action. Возвращает (return, число шагов)."""
    obs, _ = env.reset(seed=seed)
    total = 0.0
    for t in range(max_steps):
        obs, r, terminated, truncated, _ = env.step(policy(obs))
        total += r
        if terminated or truncated:
            return total, t + 1
    return total, max_steps

rng = np.random.default_rng(0)
random_policy = lambda obs: int(rng.integers(4))

results = [run_episode(GridWorldEnv(), random_policy, seed=s) for s in range(200)]
print(f"случайный агент: доходит в {np.mean([g for g, _ in results]):.0%} эпизодов "
      f"(лимит 100 шагов), в среднем за {np.mean([t for _, t in results]):.0f} шагов")

# TODO: заполните маршрут (номер клетки -> действие 0..3). Карта:
#   S....#
#   .##..#
#   ...#..
#   .#..#.
#   .#.#..
#   ....#G
route = {
    0: 1,   # клетка 0 (старт): вниз
    # ...
}
manual_policy = lambda obs: route.get(int(obs), 0)

In [ ]:
# Проверка 2: ручной маршрут доходит до цели не более чем за 12 шагов.
G, steps = run_episode(GridWorldEnv(), manual_policy)
assert G == 1.0, "ручная политика не доходит до цели"
assert steps <= 12, f"маршрут слишком длинный: {steps} шагов (кратчайший — 10)"
print(f"OK: ручная политика доходит за {steps} шагов")

### Маска действий

На лекции среда сообщала агенту в `info["action_mask"]`, какие действия не упираются в стену. Напишите функцию `action_mask(env)`, которая возвращает булев массив длины 4 для **текущего** положения `env.pos`, и политику, которая выбирает случайное действие только среди допустимых.

In [ ]:
def action_mask(env):
    """True для действий, которые сдвигают агента (не в стену и не за границу)."""
    mask = np.zeros(4, dtype=bool)
    # TODO: для каждого действия a из env.MOVES проверить, что клетка (env.pos + сдвиг) внутри поля и не стена
    raise NotImplementedError

def masked_random_policy(env, rng):
    # TODO: случайное действие среди тех, где action_mask(env) == True (np.flatnonzero пригодится)
    raise NotImplementedError

# Проверка 3: маска на старте и отсутствие ударов в стену.
env = GridWorldEnv()
env.reset(seed=0)
assert action_mask(env).tolist() == [False, True, True, False], "на старте (0, 0) доступны только ↓ и →"
rng = np.random.default_rng(0)
obs, _ = env.reset(seed=0)
bumps = 0
for _ in range(300):
    before = obs
    obs, r, terminated, truncated, _ = env.step(masked_random_policy(env, rng))
    bumps += obs == before
    if terminated:
        obs, _ = env.reset()
assert bumps == 0, f"замаскированный агент ударился в стену {bumps} раз"
print("OK: маска действий работает, ударов в стену нет")

## 4. Обёртки

Обёртка держит внутри другую среду и меняет что-то по дороге. Готовые: `TimeLimit` (обрыв по шагам), `RecordEpisodeStatistics` (return и длина в `info["episode"]`). Свою пишут, наследуя `gym.Wrapper` или один из `gym.RewardWrapper` / `gym.ObservationWrapper` / `gym.ActionWrapper`.

**Задание:** напишите `StepPenalty` — обёртку, которая вычитает `penalty` из награды на каждом шаге (плотная награда «торопись»). Наследуйте `gym.RewardWrapper` и переопределите один метод `reward(self, r)`.

In [ ]:
from gymnasium.wrappers import TimeLimit, RecordEpisodeStatistics

class StepPenalty(gym.RewardWrapper):
    def __init__(self, env, penalty=0.01):
        super().__init__(env)
        self.penalty = penalty

    def reward(self, r):
        # TODO: вернуть награду с вычтенным штрафом
        raise NotImplementedError


env = RecordEpisodeStatistics(TimeLimit(StepPenalty(GridWorldEnv(), penalty=0.01), max_episode_steps=20))
obs, _ = env.reset(seed=0)
while True:
    obs, r, terminated, truncated, info = env.step(random_policy(obs))
    if terminated or truncated:
        break
print("terminated:", terminated, " truncated:", truncated)
print("info['episode']:", {k: float(v) for k, v in info["episode"].items()})

# Проверка 3
assert truncated or terminated
assert abs(float(info["episode"]["r"]) - (1.0 * terminated - 0.01 * float(info["episode"]["l"]))) < 1e-6
print("OK: обёртки работают")

### Потенциальный shaping своими руками

Напишите обёртку `PotentialShaping(gym.Wrapper)` с наградой $R' = R + \gamma\,\Phi(s') - \Phi(s)$, где $\Phi(s) = -\text{(манхэттенское расстояние от клетки } s \text{ до цели)}$. Здесь нужен именно `gym.Wrapper`, а не `RewardWrapper`: чтобы посчитать $\Phi(s)$ **до** шага, нужно знать текущую клетку.

Подсказки: номер клетки → координаты: `divmod(obs, self.unwrapped.n_cols)`; цель: `self.unwrapped.goal`; текущее наблюдение до шага: `self.unwrapped._obs()`.

In [ ]:
class PotentialShaping(gym.Wrapper):
    def __init__(self, env, gamma=0.99):
        super().__init__(env)
        self.gamma = gamma

    def phi(self, obs):
        # TODO: минус манхэттенское расстояние от клетки obs до цели
        raise NotImplementedError

    def step(self, action):
        # TODO: phi_before -> шаг внутренней среды -> shaped = reward + gamma * phi(obs') - phi_before
        #       настоящую награду положить в info["true_reward"]
        raise NotImplementedError

# Проверка 4а: при gamma = 1 сумма подсказок по замкнутому маршруту равна нулю (телескопическая сумма).
env = PotentialShaping(GridWorldEnv(), gamma=1.0)
env.reset(seed=0)
loop = [2, 1, 0, 3]                        # → ↓ ← ↑ : вернулись в старт
total_shaped = sum(env.step(a)[1] for a in loop)
assert env.unwrapped.pos == env.unwrapped.start and abs(total_shaped) < 1e-9, "по кругу подсказки должны схлопнуться в 0"

# Проверка 4б: shaped-return эпизода = настоящий return + gamma * Phi(s_T) - Phi(s_0).
env = PotentialShaping(GridWorldEnv(), gamma=0.9)
obs0, _ = env.reset(seed=0)
route_actions = [2, 2, 2, 1, 2, 1, 2, 1, 1, 1]   # кратчайший маршрут (10 шагов) до цели на карте по умолчанию
shaped, true = 0.0, 0.0
for k, a in enumerate(route_actions):
    obs, r, terminated, truncated, info = env.step(a)
    shaped += 0.9 ** k * r; true += 0.9 ** k * info["true_reward"]
    if terminated:
        break
assert terminated, "маршрут должен приводить в цель"
assert abs(shaped - (true + 0.9 ** (k + 1) * env.phi(obs) - env.phi(obs0))) < 1e-9
print("OK: потенциальный shaping реализован верно")

## 5. Cross-Entropy на своей среде

Алгоритм из лекции 1 без изменений: ему нужны только `reset`, `step` и размеры пространств. Обучим на карте по умолчанию, потом посмотрим, что происходит на скользком полу (`slip=0.3`).

In [ ]:
def run_session(env, policy, rng, max_steps=100):
    """Один эпизод политикой-таблицей: состояния, действия, суммарная награда."""
    obs, _ = env.reset(seed=int(rng.integers(1_000_000)))
    states, actions, total = [], [], 0.0
    for _ in range(max_steps):
        a = int(rng.choice(policy.shape[1], p=policy[obs]))
        states.append(obs); actions.append(a)
        obs, r, terminated, truncated, _ = env.step(a)
        total += r
        if terminated or truncated:
            break
    return states, actions, total


def cross_entropy_method(env, n_iter=25, n_sessions=200, q=0.7, laplace=0.5, mix=0.5,
                         seed=0, max_steps=100, evaluate=None):
    """Табличный Cross-Entropy из лекции 1. evaluate(policy) -> число, если хотим отдельную метрику."""
    rng = np.random.default_rng(seed)
    n_states, n_actions = env.observation_space.n, env.action_space.n
    policy = np.ones((n_states, n_actions)) / n_actions
    log = {"mean": [], "eval": []}
    for it in range(n_iter):
        sessions = [run_session(env, policy, rng, max_steps) for _ in range(n_sessions)]
        returns = np.array([G for _, _, G in sessions])
        threshold = np.quantile(returns, q)
        elite = [s for s in sessions if s[2] >= threshold and s[2] > returns.min()]
        counts = np.full((n_states, n_actions), laplace)
        for states, actions, _ in elite:
            for s, a in zip(states, actions):
                counts[s, a] += 1
        new_policy = policy.copy()
        seen = counts.sum(axis=1) > 0
        new_policy[seen] = counts[seen] / counts[seen].sum(axis=1, keepdims=True)
        policy = mix * new_policy + (1 - mix) * policy
        log["mean"].append(returns.mean())
        if evaluate is not None:
            log["eval"].append(evaluate(policy))
    return policy, log


def show_policy(env, policy):
    for r in range(env.n_rows):
        print("  " + " ".join(env.layout[r][c] if env.layout[r][c] in "#G"
                              else env.ARROWS[int(np.argmax(policy[r * env.n_cols + c]))]
                              for c in range(env.n_cols)))

env = GridWorldEnv()
policy, log = cross_entropy_method(env, n_iter=20, n_sessions=200)
plt.plot(log["mean"], marker="."); plt.xlabel("итерация"); plt.ylabel("доля успехов"); plt.show()
show_policy(env, policy)

# Проверка 4: выученная (стохастическая) политика доходит до цели почти всегда.
rng = np.random.default_rng(5)
success = np.mean([run_session(GridWorldEnv(), policy, rng)[2] for _ in range(200)])
assert success >= 0.9, f"доля успехов {success:.2f} < 0.9"
print(f"OK: доля успехов выученной политики {success:.0%}")

In [ ]:
# Скользкий пол: действие с вероятностью 0.3 заменяется случайным.
fig, ax = plt.subplots(figsize=(7, 4))
for slip, color in [(0.0, "C0"), (0.3, "C3")]:
    for seed in range(3):
        _, log = cross_entropy_method(GridWorldEnv(slip=slip), n_iter=25, n_sessions=200, seed=seed)
        ax.plot(log["mean"], color=color, alpha=0.8, label=f"slip = {slip}" if seed == 0 else None)
ax.set_xlabel("итерация"); ax.set_ylabel("доля успехов"); ax.legend(); plt.show()

**Обсудите:** почему на скользком полу кривая ниже и шумнее, хотя цель всё та же? Что произойдёт, если сглаживание (`laplace`, `mix`) убрать?

## 6. Ловушка наивного shaping

На лекции агент с наградой «+0.1 за каждый шаг» научился гулять и не доходить до цели. Вот подсказка потоньше: **+0.5 за каждый шаг, который приблизил агента к цели**, без штрафа за удаление. Выглядит как разумная плотная награда «иди к цели». Обёртка ниже дана; ваша задача — обучить на ней Cross-Entropy с лимитом 40 шагов на карте 8×8 и **честно** оценить результат по настоящей награде.

In [ ]:
class ApproachBonus(gym.Wrapper):
    """+bonus за шаг, уменьшивший расстояние до цели; за шаг назад штрафа нет."""

    def __init__(self, env, bonus=0.5):
        super().__init__(env)
        self.bonus = bonus

    def dist(self, obs):
        r, c = divmod(int(obs), self.unwrapped.n_cols)
        gr, gc = self.unwrapped.goal
        return abs(r - gr) + abs(c - gc)

    def step(self, action):
        d_before = self.dist(self.unwrapped._obs())
        obs, reward, terminated, truncated, info = self.env.step(action)
        info["true_reward"] = reward
        if self.dist(obs) < d_before:
            reward += self.bonus
        return obs, reward, terminated, truncated, info


BIG_LAYOUT = ["S.......", ".####.#.", ".#....#.", ".#.##.#.", ".#.#..#.", ".#.#.##.", "...#....", "##.#.##G"]
T = 40
eval_env = TimeLimit(GridWorldEnv(layout=BIG_LAYOUT), max_episode_steps=T)
eval_rng = np.random.default_rng(123)

def true_success(policy, n=200):
    """Доля эпизодов, в которых агент дошёл до цели, — по настоящей среде без бонусов."""
    return np.mean([run_session(eval_env, policy, eval_rng, max_steps=T + 1)[2] for _ in range(n)])

# TODO: обучите cross_entropy_method на TimeLimit(ApproachBonus(GridWorldEnv(layout=BIG_LAYOUT)), max_episode_steps=T)
#       с n_iter=25, n_sessions=200, max_steps=T + 1, evaluate=true_success; сохраните policy и log
policy_trap, log_trap = None, None

# TODO: нарисуйте два графика: log_trap["mean"] (награда с бонусами) и log_trap["eval"] (настоящая доля успехов)

# TODO: прогоните один эпизод выученной политикой по настоящей среде и напечатайте список посещённых клеток

In [ ]:
# Проверка 5: агент собирает бонусов больше, чем даёт кратчайший путь (14 приближений × 0.5 + 1 = 8),
# значит, он научился «приближаться» несколько раз к одной и той же цели — ходить туда-сюда.
assert log_trap["mean"][-1] > 8.5, "ожидали return по shaped-награде выше 8.5: агент должен был найти лазейку"
print(f"OK: return с бонусами {log_trap['mean'][-1]:.1f} при максимуме честного пути 8.0; "
      f"настоящая доля успехов {log_trap['eval'][-1]:.0%}")

<details>
<summary>Что здесь произошло</summary>

Шаг к цели даёт +0.5, шаг от цели — 0. Пара «вперёд-назад» стоит +0.5 и ничего не стоит: агент может «приближаться» к цели бесконечно, не приходя в неё. С лимитом 40 шагов колебание в коридоре приносит до 10, а честный путь — 8. Cross-Entropy находит смесь: часть эпизодов колеблется, часть доходит. Потенциальный shaping из части 4 этой лазейки не оставляет: шаг назад стоит ровно столько, сколько принёс шаг вперёд.

</details>

## 7. Частичная наблюдаемость на CartPole

На лекции мы прятали от агента угловую скорость шеста. Повторите это обёрткой над наблюдением: `HideVelocity(gym.ObservationWrapper)` оставляет из четырёх чисел CartPole только положение тележки и угол (индексы 0 и 2), а `FrameStackObservation` из Gymnasium возвращает историю из двух кадров. Не забудьте поправить `observation_space`: алгоритмы читают из него размер входа.

In [ ]:
from gymnasium.wrappers import FrameStackObservation

class HideVelocity(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        # TODO: self.observation_space = Box из компонент 0 и 2 исходного пространства (dtype=np.float32)
        raise NotImplementedError

    def observation(self, obs):
        # TODO: вернуть obs[[0, 2]] как float32
        raise NotImplementedError


def evaluate_env(make_env, policy, n_episodes=30):
    """policy(obs) -> action. Средний return по эпизодам с сидами 0..n-1."""
    env = make_env()
    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=ep)
        total = 0.0
        while True:
            obs, r, terminated, truncated, _ = env.step(policy(obs))
            total += r
            if terminated or truncated:
                break
        returns.append(total)
    env.close()
    return float(np.mean(returns))

blind = lambda: HideVelocity(gym.make("CartPole-v1"))
stacked = lambda: FrameStackObservation(HideVelocity(gym.make("CartPole-v1")), stack_size=2)

# TODO: две политики: angle_only(obs) — толкать в сторону наклона по одному кадру;
#       two_frames(obs) — по разности углов между кадрами obs[1] и obs[0]
angle_only = None
two_frames = None

# Проверка 6
env = blind()
obs, _ = env.reset(seed=0)
assert env.observation_space.shape == (2,) and obs.shape == (2,) and obs.dtype == np.float32
assert env.observation_space.contains(obs)
assert stacked().observation_space.shape == (2, 2)
r_blind, r_stack = evaluate_env(blind, angle_only), evaluate_env(stacked, two_frames)
print(f"только угол: {r_blind:.0f}   два кадра: {r_stack:.0f}")
assert r_stack > 100 > r_blind, "стек из двух кадров должен возвращать скорость и результат"
print("OK: частичная наблюдаемость вылечена стеком кадров")

**Обсудите:** какую награду вы бы добавили обёрткой, чтобы агент на скользком полу держался подальше от стен, — и как проверить, что она не сломала настоящую цель? Что случится с политикой из части 7, если стек сделать из 4 кадров, как в DQN, — станет ли лучше?

## 8. Что дальше

* **Домашнее задание** (`../homework/homework.ipynb`): своя среда «управление запасами» как `gym.Env` со случайным спросом, две версии награды и сравнение обучения Cross-Entropy, формальное описание среды как MDP; бонус — скрытый день недели.
* **Неделя 3**: методы, использующие структуру MDP, — ценность состояния, уравнение Беллмана, динамическое программирование, Monte-Carlo и TD-обучение.